# StereoQueerEval 2027 — Task B: Class-Aware Multi-Head Cross-Attention (End-to-End)

Notebook này thiết lập quy trình huấn luyện **End-to-End** cho **Task B (Hate Speech Classification - 3 classes: `no`, `yes_implicit`, `yes_explicit`)** sử dụng kiến trúc tùy chỉnh nâng cao:

```
[INPUT SEQUENCE]
Title: <T> ... </T> | Description: <D> ... </D> | Comment: <C> ... </C>
                                   │
                                   ▼
┌───────────────────────────────────────────────────────────────────────────────┐
│ LAYER 0: ENCODER & EXPLICIT ROLE INJECTION                                    │
│  Input Token IDs  [B, S] ──────► mmBERT Encoder ────► H_mmBERT  [B, S, 768]   │
│  Role IDs         [B, S] ──────► Role Embeddings ───► E_role    [B, S, 768]   │
│                               H_final = H_mmBERT + E_role ───┴─► [B, S, 768]  │
└──────────────────────────────────────┬────────────────────────────────────────┘
                                       ▼
┌───────────────────────────────────────────────────────────────────────────────┐
│ LAYER 1: CLASS-AWARE MULTI-HEAD CROSS-ATTENTION (MHCA)                        │
│  Learned Queries: Q_base = [q_Explicit, q_Implicit, q_NonHate] ∈ [3, 768]    │
│  A = Softmax(Q·Kᵀ / √d_h)       ∈ [B, 3, S]  (Interpretability Attention Map) │
│  Z = LayerNorm(Q + Dropout(MHCA(Q, K, V)))    ──► Tensor Z ∈ [B, 3, 768]      │
└──────────────────────────────────────┬────────────────────────────────────────┘
                                       ▼
┌───────────────────────────────────────────────────────────────────────────────┐
│ LAYER 2: QUERY INTERACTION LAYER (MHSA) [Ablation Hypothesis H2]              │
│  Self-Attention: Explicit ↔ Implicit ↔ NonHate                                │
│  Z' = LayerNorm(Z + Dropout(MHSA(Q=Z, K=Z, V=Z))) ──► Tensor Z' ∈ [B, 3, 768] │
└──────────────────────────────────────┬────────────────────────────────────────┘
                                       ▼
┌───────────────────────────────────────────────────────────────────────────────┐
│ LAYER 3: SHARED SCORING HEAD & TASK C BRIDGE                                  │
│  z'_c [B, 768] ──► Shared Head f_θ ──► Raw Scores s ∈ [B, 3]                  │
│  Probabilities p = Softmax(s)                                                 │
│  Task C Bridge: h_B = ∑_c (p_c · z'_c) ∈ [B, 768]                            │
└───────────────────────────────────────────────────────────────────────────────┘
```

---
### Quy trình trong notebook:
1. **Environment Setup & Git Clone / Pull repository**
2. **Cài đặt thư viện (PyTorch, Transformers, Accelerate, v.v.)**
3. **Chuẩn bị dữ liệu TSV (hoặc sinh sample data kiểm tra)**
4. **Kiểm tra Role IDs & Tokenizer Aligned Data Loader**
5. **Khởi tạo mô hình `TaskBClassAwareAttentionModel`**
6. **Training Loop 2-Phase (Phase 1: Frozen Backbone -> Phase 2: Fine-Tuning Last N Layers)**
7. **Ablation Study: Bật / Tắt Layer 2 MHSA (`use_query_interaction`)**
8. **Đánh giá chi tiết (Macro-F1, Per-class F1, Confusion Matrix, Error Analysis)**
9. **Visualizing Attention Maps (Hiển thị từ nào trigger Explicit / Implicit / NonHate)**
10. **Xuất kết quả dự đoán và trích xuất Task C Bridge representations $h_B$**

## 1. Clone hoặc Cập nhật Mã Nguồn từ GitHub

Thay thế URL repo của bạn ở bên dưới nếu đang chạy trên Google Colab / Kaggle / Server riêng.

In [ ]:
# Khai báo thông tin repository (thay bằng URL GitHub của bạn)
import os
import sys

REPO_URL = "https://github.com/your-username/stereoqueer-pipeline.git"
REPO_DIR = "stereoqueer-pipeline"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    print(f"Repository directory '{REPO_DIR}' already exists.")
    if os.path.isdir(REPO_DIR):
        %cd {REPO_DIR}
        !git pull

# Thêm root vào sys.path để import pipeline package
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Current Working Directory:", os.getcwd())
!ls -la

## 2. Cài đặt Thư viện Phụ thuộc (Dependencies)

In [ ]:
!pip install -q torch transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("Memory Allocated:", round(torch.cuda.memory_allocated(0)/1024**3, 2), "GB")

## 3. Chuẩn bị Dữ liệu Huấn luyện
- Nếu bạn có file chính thức: đặt vào thư mục `data/`:
  - `data/StereoQueerEval_EN_training.tsv`
  - `data/StereoQueerEval_IT_training.tsv`
  - `data/StereoQueerEval_NL_training.tsv`
- Nếu chưa có, script bên dưới sẽ tự động sinh dữ liệu mẫu giả lập đầy đủ các cột và nhãn.

In [ ]:
os.makedirs("data", exist_ok=True)
os.makedirs("checkpoints_task_b", exist_ok=True)

# Kiểm tra file TSV có sẵn không
tsv_files = [os.path.join("data", f) for f in os.listdir("data") if f.endswith(".tsv")]

if not tsv_files:
    print("Chưa phát hiện file TSV. Đang tạo sample dataset để kiểm thử pipeline...")
    !python generate_sample_data.py
    tsv_files = [os.path.join("data", f) for f in os.listdir("data") if f.endswith(".tsv")]

print(f"Đã tìm thấy {len(tsv_files)} file TSV:")
for f in sorted(tsv_files):
    print(" -", f)

## 4. Load Data & Anti-Leakage Video Splitting (`GroupShuffleSplit`)
Đảm bảo các comment từ cùng một video YouTube (`yt_title`) không bị rò rỉ (leakage) giữa tập train và validation.

In [ ]:
import pandas as pd
import numpy as np
from pipeline.config import PipelineConfig, HATE_CLASSES, HATE2IDX, IDX2HATE
from pipeline.data import DataPipeline
from pipeline.task_b_data import TaskBRoleDataset
from pipeline.models.task_b_class_aware import (
    TaskBClassAwareAttentionModel,
    ROLE_PAD, ROLE_TITLE, ROLE_DESC, ROLE_COMMENT
)

config = PipelineConfig(
    task="stereoqueer",
    target_task="hs", # Task B
    model_type="task_b_class_aware",
    mmbert_model_name="jhu-clsp/mmbert-base",
    batch_size=32,
    max_length=256,
    learning_rate=1e-4,
    two_phase=True,
    freeze_phase_epochs=10,
    unfreeze_phase_epochs=10,
    unfreeze_layers=2,
    unfreeze_lr=2e-5,
    head_unfreeze_lr=5e-5,
    patience=5,
    output_dir="checkpoints_task_b"
)

data_pipeline = DataPipeline(config)
data_pipeline.load_data(tsv_files)
df_train, df_val = data_pipeline.split_data(test_size=0.15, random_state=42)

print(f"Train samples: {len(df_train)} | Val samples: {len(df_val)}")
print("Phân bố nhãn Task B trên tập Train:")
print(df_train['hate_speech'].value_counts())
print("\nPhân bố nhãn Task B trên tập Val:")
print(df_val['hate_speech'].value_counts())

## 5. Tokenizer & Explicit Role Injection Dataset (`TaskBRoleDataset`)

Tokenize từng đoạn (`Title`, `Description`, `Comment`) và gán nhãn `Role ID` tương ứng để đưa vào **Layer 0** của mô hình.

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

print(f"Loading ModernBERT/mmBERT tokenizer: {config.mmbert_model_name}...")
tokenizer = AutoTokenizer.from_pretrained(config.mmbert_model_name)

train_ds = TaskBRoleDataset(df_train, tokenizer, max_len=config.max_length)
val_ds = TaskBRoleDataset(df_val, tokenizer, max_len=config.max_length)

train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)

# Kiểm tra 1 mẫu đầu vào và role IDs
sample_input_ids, sample_mask, sample_roles, sample_st, sample_hs, sample_tg = train_ds[0]
print("Sample Input IDs shape:", sample_input_ids.shape)
print("Sample Role IDs shape:", sample_roles.shape)
print("Role IDs unique counts:", torch.bincount(sample_roles))
print("Sample Label HS (0=no, 1=implicit, 2=explicit):", sample_hs.item(), f"({IDX2HATE[sample_hs.item()]})")

# Giải mã 20 token đầu tiên kèm vai trò
tokens = tokenizer.convert_ids_to_tokens(sample_input_ids[:25])
roles = sample_roles[:25].tolist()
role_names = {ROLE_PAD: 'PAD/SEP', ROLE_TITLE: 'TITLE', ROLE_DESC: 'DESC', ROLE_COMMENT: 'COMMENT'}
print("\nFirst 25 tokens with role mapping:")
for t, r in zip(tokens, roles):
    print(f"  {t:<15} -> {role_names.get(r, 'UNK')}")

## 6. Khởi tạo Kiến trúc Task B: Class-Aware Cross-Attention

Mô hình bao gồm:
- **Layer 0**: mmBERT backbone + `role_embeddings` ($E_{role} \in [B, S, 768]$)
- **Layer 1**: Class-Aware Cross-Attention với 3 truy vấn học được $Q_{base} = [q_{NonHate}, q_{Implicit}, q_{Explicit}] \in [3, 768]$
- **Layer 2**: Query Interaction Layer (MHSA giữa 3 nhãn) [H2 Toggle]
- **Layer 3**: Shared Scoring Head $f_\theta$ sinh điểm raw $s \in [B, 3]$ và Task C bridge $h_B$

In [ ]:
from transformers import AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

print(f"Loading backbone: {config.mmbert_model_name}...")
backbone = AutoModel.from_pretrained(config.mmbert_model_name)

model = TaskBClassAwareAttentionModel(
    mmbert_model=backbone,
    d_model=config.mmbert_dim,
    num_heads=config.num_heads,
    dropout=config.dropout,
    use_query_interaction=True # Layer 2 MHSA enabled
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {total_params:,} | Trainable: {trainable_params:,}")

# Kiểm tra thử 1 forward pass
dummy_ids = sample_input_ids.unsqueeze(0).to(device)
dummy_mask = sample_mask.unsqueeze(0).to(device)
dummy_roles = sample_roles.unsqueeze(0).to(device)

with torch.no_grad():
    logits, h_B, attn_map = model(dummy_ids, dummy_mask, dummy_roles, return_attention_map=True)

print("Forward Output Logits shape:", logits.shape)       # [1, 3]
print("Task C Bridge h_B shape:", h_B.shape)             # [1, 768]
print("Attention Map shape:", attn_map.shape)             # [1, 3, 256]
print("Probabilities:", torch.softmax(logits, dim=-1).cpu().numpy())

## 7. Huấn Luyện 2-Phase End-to-End

- **Phase 1**: Đóng băng mmBERT (`requires_grad = False`), tập trung tối ưu Role Embeddings, Learned Class Queries, Cross-Attention (MHCA), và Shared Head.
- **Phase 2**: Mở khóa $N$ layer cuối cùng của mmBERT với tốc độ học nhỏ (discriminative fine-tuning LR `2e-5`) để tinh chỉnh biểu diễn ngôn ngữ đa ngữ.

In [ ]:
from pipeline.task_b_trainer import TaskBTrainer

trainer = TaskBTrainer(
    model=model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    df_val=df_val
)

print("Bắt đầu huấn luyện mô hình Task B...")
results = trainer.train()

print("\nHuấn luyện hoàn tất!")
print("Best Checkpoint File:", results['checkpoint_path'])
print("Final Metrics:", results['final_metrics'])

## 8. Trực Quan Hóa Quá Trình Huấn Luyện (Loss & Macro-F1 Curves)

In [ ]:
import matplotlib.pyplot as plt

history = results['history']
if history:
    epochs = [h['epoch'] for h in history]
    train_loss = [h['train_loss'] for h in history]
    val_loss = [h['val_loss'] for h in history]
    macro_f1 = [h['hs_macro_f1'] for h in history]

    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_loss, label='Train Loss', marker='o')
    plt.plot(epochs, val_loss, label='Val Loss', marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('Cross Entropy Loss')
    plt.title('Training & Validation Loss')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, macro_f1, label='Val Macro-F1', color='green', marker='^')
    plt.xlabel('Epoch')
    plt.ylabel('Macro-F1')
    plt.title('Task B Macro-F1 Over Epochs')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("No history recorded.")

## 9. Đánh Giá Chi Tiết & Ma Trận Nhầm Lẫn (Confusion Matrix)

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Đọc file dự đoán đã lưu từ checkpoint tốt nhất
pred_csv_path = os.path.join(config.output_dir, "task_b_val_predictions.csv")
if os.path.exists(pred_csv_path):
    df_preds = pd.read_csv(pred_csv_path)
    y_true = df_preds['hate_speech'].tolist()
    y_pred = df_preds['pred_hate_speech'].tolist()
    
    labels = ['no', 'yes_implicit', 'yes_explicit']
    print("=== CLASSIFICATION REPORT (Task B) ===")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))
    
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True Gold Label')
    plt.title('Confusion Matrix - Task B Class-Aware MHCA')
    plt.show()
else:
    print(f"File {pred_csv_path} not found.")

## 10. Chạy Nghiên Cứu Triệt Tiêu (Ablation Study Hypothesis H2)

So sánh hiệu năng khi **bật** vs **tắt** Layer 2 Query Interaction (MHSA inter-label attention):
- **H2 Full**: $Z' = \text{MHSA}(Z)$
- **H2 Ablated**: $Z' = Z$ (không có tương tác giữa các query nhãn)

In [ ]:
print("Để chạy thử nghiệm Ablation Study H2 không có Layer 2 MHSA, bạn có thể thực thi lệnh:")
print("python train.py --target_task hs --embed_source mmbert --model task_b_class_aware --no_query_interaction --output_dir checkpoints_task_b_no_h2")

# Hoặc chạy trực tiếp từ notebook:
# !python train.py --target_task hs --embed_source mmbert --model task_b_class_aware --no_query_interaction --freeze_epochs 5 --unfreeze_epochs 5

## 11. Interpretability: Trực Quan Hóa Class-Conditioned Attention Map

Quan sát xem mỗi query ($q_{Explicit}, q_{Implicit}, q_{NonHate}$) chú ý vào những từ nào trong bình luận và ngữ cảnh YouTube.

In [ ]:
model.eval()
# Chọn 1 mẫu có hate speech để phân tích giải thích
test_idx = 0
for i in range(len(val_ds)):
    if val_ds[i][4].item() != 0: # Tìm mẫu implicit hoặc explicit
        test_idx = i
        break

inp_ids, att_mask, r_ids, _, true_hs, _ = val_ds[test_idx]
with torch.no_grad():
    out_s, out_hB, attn_weights = model(
        inp_ids.unsqueeze(0).to(device),
        att_mask.unsqueeze(0).to(device),
        r_ids.unsqueeze(0).to(device),
        return_attention_map=True
    )

tokens = tokenizer.convert_ids_to_tokens(inp_ids.tolist())
# Lấy valid tokens (bỏ padding)
valid_len = int(att_mask.sum().item())
tokens = tokens[:valid_len]
attn_map_np = attn_weights[0, :, :valid_len].cpu().numpy() # [3, valid_len]

plt.figure(figsize=(14, 3.5))
sns.heatmap(
    attn_map_np,
    cmap="YlOrRd",
    xticklabels=[t if len(t) < 12 else t[:10] + '..' for t in tokens],
    yticklabels=['NonHate Query', 'Implicit Query', 'Explicit Query']
)
plt.xticks(rotation=90, fontsize=8)
plt.title(f'Attention Distribution Across Tokens (True Label: {IDX2HATE[true_hs.item()]})')
plt.show()

## 12. Task C Representation Bridge ($h_B$ Extraction)

Xuất biểu diễn $h_B = \sum_c (p_c \cdot z'_c) \in [B, 768]$ để làm đầu vào cho bài toán nhận diện đối tượng đích (Task C: Target Identity & Scope).

In [ ]:
print("Trích xuất Task C Bridge representations h_B cho tập Validation...")
model.eval()
h_B_list = []

with torch.no_grad():
    for batch in val_loader:
        i_ids, a_mask, r_ids, _, _, _ = batch
        _, h_B, _ = model(i_ids.to(device), a_mask.to(device), r_ids.to(device))
        h_B_list.append(h_B.cpu().numpy())

h_B_all = np.concatenate(h_B_list, axis=0)
print(f"Extracted h_B matrix shape: {h_B_all.shape} ([N_val, 768])")

# Lưu vector làm feature cache cho Task C
bridge_save_path = os.path.join(config.output_dir, "val_task_c_bridge_h_B.npy")
np.save(bridge_save_path, h_B_all)
print(f"Saved bridge vectors to: {bridge_save_path}")
print("Hoàn tất toàn bộ quy trình Task B!")